# ChakraNet GPU Verification (Colab Edition)
This notebook is perfectly tuned to find your files in Google Drive automatically, patch the CPU lock, and run the verification using Colab's T4 GPU.

In [ ]:
# 1. Install required dependencies
!pip install -q ultralytics timm einops albumentations

In [ ]:
import os
import shutil
from google.colab import drive

# 2. Mount Google Drive
drive.mount('/content/drive')

# Search for the weights and data zip
print("\nHunting for the files in Google Drive...")
target_weight = 'chakra_transformer_best.pth'
target_yolo = 'best.pt'
target_zip = 'chakramodel_data_scripts.zip'

weight_path = None
yolo_path = None
zip_path = None

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if target_weight in files and weight_path is None:
        weight_path = os.path.join(root, target_weight)
        yolo_path = os.path.join(root, target_yolo)
    if target_zip in files and zip_path is None:
        zip_path = os.path.join(root, target_zip)

if not weight_path:
    print(f"\n❌ CRITICAL: Could not find '{target_weight}' in Google Drive. Make sure it's uploaded!")
else:
    print(f"\n✅ Found weights at: {weight_path}")
    os.makedirs('/content/weights', exist_ok=True)
    shutil.copy(weight_path, '/content/weights/chakra_transformer_best.pth')
    if os.path.exists(yolo_path):
        shutil.copy(yolo_path, '/content/weights/best.pt')
        print("✅ Copied YOLO weights too!")
        
if not zip_path:
    print(f"\n❌ CRITICAL: Could not find '{target_zip}' in Google Drive. Make sure it's uploaded!")
else:
    print(f"\n✅ Found data zip at: {zip_path}")
    shutil.copy(zip_path, '/content/chakramodel_data_scripts.zip')


In [ ]:
# 3. Unzip the code and dataset
!unzip -q -o /content/chakramodel_data_scripts.zip -d /content/

# CRITICAL FIX: The verify_strict.py script currently forces CPU mode. 
# We need to remove the CPU mock so it uses Colab's T4 GPU!
script_path = '/content/src/verify_strict.py'
with open(script_path, 'r') as f:
    content = f.read()

# Remove the line that forces CPU
content = content.replace('torch.cuda.is_available = lambda: False', '# torch.cuda.is_available = lambda: False (Removed to use Colab GPU)')

with open(script_path, 'w') as f:
    f.write(content)
print("✅ Patched verify_strict.py to use the Colab GPU!")

In [ ]:
# 4. Run the strict verification on Colab's GPU!
!python /content/src/verify_strict.py